# WP35 — Meta-Learning from Failure (v0.4)
## FailureMemory · ErrorClassifier · ConditionedSearcher

Demonstrates **WP35**: persisting tactic failure patterns across theorems and conditioning the proof-search prior on difficulty-specific failure rates.

Runtime: **~2 min** (no GPU, no Lean required)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
class MockLean:
    _using_mock = True
    def use(self, code):
        if 'sorry' in code: return "Proof contains 'sorry'"
        if 'ring' in code and ':= by' in code: return None
        return 'unsolved goals'
lean = MockLean()
print('LeanTool mock ready')

In [ ]:
from prometheus.wp35_failure_memory import ErrorClassifier, FailureMemory

ec = ErrorClassifier()
test_errors = [
    'type mismatch: expected Nat but got Int',
    'unsolved goals',
    'unknown tactic blah',
    "declaration uses 'sorry'",
    'deterministic timeout',
    'some other weird error',
]
print('ErrorClassifier results:')
for err in test_errors:
    cls = ec.classify(err)
    print(f'  {cls:<18}  <- "{err[:50]}"')

In [ ]:
from prometheus.wp35_failure_memory import FailureMemory
from prometheus.wp34_proof_tree_search import _DEFAULT_LEAN4_PRIOR

mem = FailureMemory(penalty_weight=0.70, min_observations=3)

# Seed failures: simp fails on hard theorems 5 times
for _ in range(5):
    mem.record('simp', 'unsolved goals', 'hard', depth=1)
for _ in range(3):
    mem.record('omega', 'unsolved goals', 'hard', depth=2)
# simp succeeds on easy theorems
for _ in range(5):
    mem.record_success('simp', 'easy')

print('FailureMemory state:')
print(f'  Total failure records: {mem.to_dict()["n_failure_records"]}')
print(f'  simp failure_rate(hard): {mem.failure_rate("simp", "hard"):.2f}')
print(f'  simp failure_rate(easy): {mem.failure_rate("simp", "easy"):.2f}')
print(f'  omega failure_rate(hard): {mem.failure_rate("omega", "hard"):.2f}')
print()
print('Conditioned prior (hard difficulty) — simp should be penalised:')
base = _DEFAULT_LEAN4_PRIOR
cond = mem.conditioned_prior('hard', base)
for tac in ['simp','ring','omega','rfl']:
    b = base.score(tac)
    c = cond.score(tac)
    diff = c - b
    print(f'  {tac:<12} base={b:.3f}  cond={c:.3f}  delta={diff:+.3f}')

In [ ]:
from prometheus.wp35_failure_memory import ConditionedSearcher

searcher = ConditionedSearcher(lean_tool=lean, max_nodes=80, branching_factor=5)
THEOREMS_BY_DIFF = {
    'easy':   ['theorem add_zero (n : Nat) : n + 0 = n',
                'theorem zero_add (n : Nat) : 0 + n = n'],
    'medium': ['theorem add_comm_nat (a b : Nat) : a + b = b + a',
                'theorem mul_comm_nat (a b : Nat) : a * b = b * a'],
    'hard':   ['theorem add_assoc_nat (a b c : Nat) : a + b + c = a + (b + c)',
                'theorem mul_add_nat (a b c : Nat) : a * (b + c) = a * b + a * c'],
}
all_records = []
for diff, thms in THEOREMS_BY_DIFF.items():
    for thm in thms:
        r = searcher.search(thm, difficulty=diff)
        all_records.append((diff, r))
        ok = 'PASS' if r.success else 'FAIL'
        print(f'  [{diff:<6}] [{ok}] nodes={r.nodes_expanded}  {thm[:55]}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
diffs = ['easy','medium','hard']
diff_colors = {'easy':'#4CAF50','medium':'#FF9800','hard':'#F44336'}

# Panel A: failure rates per tactic per difficulty
ax = axes[0]
show_tactics = ['simp','ring','omega','rfl']
for i, diff in enumerate(diffs):
    rates = [searcher.failure_memory.failure_rate(t, diff) for t in show_tactics]
    ax.bar([j + i*0.28 for j in range(len(show_tactics))], rates, 0.26,
           label=diff, color=diff_colors[diff], alpha=0.85, edgecolor='black')
ax.set_xticks(range(len(show_tactics))); ax.set_xticklabels(show_tactics)
ax.set_ylabel('Failure Rate'); ax.set_ylim(0, 1)
ax.set_title('Tactic Failure Rates per Difficulty', fontweight='bold'); ax.legend()

# Panel B: success per difficulty
ax2 = axes[1]
for i, diff in enumerate(diffs):
    recs = [r for d, r in all_records if d == diff]
    n_ok = sum(r.success for r in recs)
    ax2.bar(i, n_ok / max(len(recs),1), color=diff_colors[diff], edgecolor='black', alpha=0.85)
ax2.set_xticks(range(3)); ax2.set_xticklabels(diffs)
ax2.set_ylabel('Success Rate'); ax2.set_ylim(0, 1.15)
ax2.set_title('Proof Success Rate by Difficulty', fontweight='bold')

# Panel C: error class distribution
ax3 = axes[2]
dist = searcher.failure_memory.error_class_distribution()
if dist:
    ax3.bar(dist.keys(), dist.values(), color='#9C27B0', edgecolor='black', alpha=0.85)
ax3.set_ylabel('Count'); ax3.set_title('Error Class Distribution', fontweight='bold')

fig.suptitle('WP35: Meta-Learning from Failure', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp35_failure_memory.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved wp35_failure_memory.png')

In [ ]:
from prometheus.wp35_failure_memory import verify_wp35_exit_criteria
criteria = verify_wp35_exit_criteria(searcher, [r for _,r in all_records])
print('WP35 Exit Criteria Verification'); print('='*55)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()):
    print('\nAll WP35 exit criteria satisfied.')

---
## Conclusions

**WP35** adds cross-theorem failure memory:
- `ErrorClassifier` maps raw Lean errors to named classes
- `FailureMemory` persists (tactic, difficulty) failure rates
- `ConditionedSearcher` builds a penalised prior before each search

### Foundation for WP36
WP36 adds a transformer head that conditions on the *history of synthesis actions* — analagous to WP35's conditioning on tactic failure history.